In [1]:
import os
import faiss
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from groq import Groq

### Loading dataset

In [2]:
print("Loading dataset from Hugging face [wikipedia passages formatted for RAG]")
dataset = load_dataset("rag-datasets/rag-mini-wikipedia","text-corpus")
print(dataset)

Loading dataset from Hugging face [wikipedia passages formatted for RAG]


DatasetDict({
    passages: Dataset({
        features: ['passage', 'id'],
        num_rows: 3200
    })
})


In [3]:
first_row = dataset['passages'][0]
print('1st row:',first_row.keys())
print('sample passages text:',first_row['passage'])

1st row: dict_keys(['passage', 'id'])
sample passages text: Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.


In [4]:
raw_passages = dataset['passages']['passage']
print(f"Loaded {len(raw_passages)} base passges from dataset")

Loaded 3200 base passges from dataset


### Inspect & chunking 

In [5]:
print("Inspect & Chunk passages")
avg_len = sum(len(p) for p in raw_passages) / len(raw_passages)
print(avg_len,"charcters in each passages")

Inspect & Chunk passages
389.848125 charcters in each passages


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len
)
chunks = []
for t in raw_passages:
    split_text = splitter.split_text(t)
    chunks.extend(split_text)
print(f'total chunks created:{len(chunks)}')

total chunks created:4492


### Generating embedding

In [7]:
print('Generating embedding')
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device = "cpu")

# convert chunks to diemnsional vectors
embeddings = embedding_model.encode(chunks, show_progress_bar = True, convert_to_numpy = True)

# Normalize vectors(consine similarity in faiss)
faiss.normalize_L2(embeddings)
print(f'Embeddings matrix shape: {embeddings.shape}')

Generating embedding


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Embeddings matrix shape: (4492, 384)


### Build faiss index & save to disk 

In [8]:
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

os.makedirs('vector_store',exist_ok = True)
faiss.write_index(index, 'vector_store/index.faiss')
np.save('vector_store/chunks.npy', np.array(chunks, dtype= object))

print(f'successfully processed & stored {len(chunks)} chunks')
print('faiss index saved at: vector_store/index.faiss')
print('text chunks saved at: vector_store/chunks.npy')

successfully processed & stored 4492 chunks
faiss index saved at: vector_store/index.faiss
text chunks saved at: vector_store/chunks.npy


### Retriving & Generating

In [9]:
# loading vector index & chunks 
index = faiss.read_index('vector_store/index.faiss')
chunks = np.load('vector_store/chunks.npy', allow_pickle = True)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device = "cpu")
groq_client = Groq(api_key = "YOUR_API_KEY")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
def retrieve_chunks(query : str, top_k : int = 3):
    query_vector = embedding_model.encode([query], convert_to_numpy = True)
    faiss.normalize_L2(query_vector)
    score, indices = index.search(query_vector, top_k)

    retrieved = []
    for s, i in zip(score[0], indices[0]):
        retrieved.append({
            "score":float(s),
            "text": chunks[i]
        })
    return retrieved

In [12]:
def ask_rag(question: str, top_k : int = 3):
    result = retrieve_chunks(question, top_k = top_k)
    context_text = '\n\n'.join([r['text'] for r in result])
    
    prompt = f"Answer the question using only the context below.If the answer isn't in the context, say so.\n\nContext: {context_text}\n\nQuestion: {question}"
    completion = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)

    answer = completion.choices[0].message.content
    print(f"QUESTION: {question}")
    print("\n--- RETRIEVED CONTEXT CHUNKS ---")
    for i, res in enumerate(result, 1):
        print(f"[Chunk {i}] (Similarity Score: {res['score']})")
        print(res["text"])
    print("GENERATED ANSWER:")
    print(answer)

In [13]:
# testing complete RAG pipeline
user_question = "what is the anatomy of a beetle?"
ask_rag(user_question, top_k = 3)

QUESTION: what is the anatomy of a beetle?

--- RETRIEVED CONTEXT CHUNKS ---
[Chunk 1] (Similarity Score: 0.7440573573112488)
The general anatomy of beetles is quite uniform, although specific organs and appendages may vary greatly in appearance and function between the many families in the order. Like all insects, beetles' bodies are divided into three sections: the head, the thorax, and the abdomen. When viewed from below, the thorax is that part from which all three pairs of legs and both pairs of wings arise. The abdomen is everything posterior to the thorax. When viewed from above, most beetles appear to have
[Chunk 2] (Similarity Score: 0.6456075310707092)
*  The Beetle Ring - A group of websites about beetles (Coleoptera).
[Chunk 3] (Similarity Score: 0.6414692997932434)
viewed from above, most beetles appear to have three clear sections, but this is deceptive: on the beetle's upper surface, the middle "section" is a hard plate called the pronotum, which is only the front part o

In [14]:
user_question = "Where do legs and wings attach on a beetle?"
ask_rag(user_question, top_k = 3)

QUESTION: Where do legs and wings attach on a beetle?

--- RETRIEVED CONTEXT CHUNKS ---
[Chunk 1] (Similarity Score: 0.7565822005271912)
to move the hind flight wings. A beetle's flight wings are crossed with veins and are folded after landing, often along these veins, and are stored below the elytra.
[Chunk 2] (Similarity Score: 0.6761083602905273)
modified for swimming and often bear rows of long hairs to aid this purpose. Other beetles have fossorial legs that are widened and often spined for digging. Species with such adaptations are found among the scarabs, ground beetles, and clown beetles (family Histeridae). The hind legs of some beetles, such as flea beetles (within Chrysomelidae) and flea weevils (within Curculionidae), are enlarged and designed for jumping.
[Chunk 3] (Similarity Score: 0.6592586040496826)
Beetles are generally characterised by a particularly hard exoskeleton and hard forewings (elytra). The beetle's exoskeleton is made up of numerous plates called sclerites,

In [15]:
user_question = "What is a pronotum?"
ask_rag(user_question, top_k = 3)

QUESTION: What is a pronotum?

--- RETRIEVED CONTEXT CHUNKS ---
[Chunk 1] (Similarity Score: 0.3594795763492584)
Spheniscus mendiculus
[Chunk 2] (Similarity Score: 0.34941986203193665)
Spheniscus demersus
[Chunk 3] (Similarity Score: 0.348178505897522)
than the tergites; antennae with 11 or fewer segments; and terminal genitalic appendages retracted into the abdomen and invisible at rest.
GENERATED ANSWER:
The answer isn’t in the context.


In [16]:
user_question = "What is the average lifespan of a beetle?"
ask_rag(user_question, top_k = 3)

QUESTION: What is the average lifespan of a beetle?

--- RETRIEVED CONTEXT CHUNKS ---
[Chunk 1] (Similarity Score: 0.7296607494354248)
As with all endopterygotes, beetle larvae pupate, and from this pupa emerges a fully formed, sexually mature adult beetle, or imago. Adults have an extremely variable lifespan, from weeks to years, depending on the species.
[Chunk 2] (Similarity Score: 0.5578525066375732)
varies between species but can be as long as several years.
[Chunk 3] (Similarity Score: 0.5361716151237488)
Beetles are endopterygotes with complete metamorphosis.
GENERATED ANSWER:
The context does not provide a specific average lifespan for beetles. It only states that adult beetles can live from a few weeks to several years, depending on the species.


In [17]:
user_question = "How do beetles protect their flight wings?"
ask_rag(user_question, top_k = 3)

QUESTION: How do beetles protect their flight wings?

--- RETRIEVED CONTEXT CHUNKS ---
[Chunk 1] (Similarity Score: 0.6945784091949463)
to move the hind flight wings. A beetle's flight wings are crossed with veins and are folded after landing, often along these veins, and are stored below the elytra.
[Chunk 2] (Similarity Score: 0.6900066137313843)
Beetles are generally characterised by a particularly hard exoskeleton and hard forewings (elytra). The beetle's exoskeleton is made up of numerous plates called sclerites, separated by thin sutures. This design creates the armoured defences of the beetle while maintaining flexibility. The elytra are not used for flight, but tend to cover the hind part of the body and protect the second pair of wings (alae). The elytra must be raised in order to move the hind flight wings. A beetle's flight
[Chunk 3] (Similarity Score: 0.6850190162658691)
Beetles and their larvae have a variety of strategies to avoid being attacked by predators or parasitoid